In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from PIL import Image
from wordcloud import WordCloud
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer

from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import get_scheduler

!pip install evaluate
import evaluate

import torch

In [ ]:
# Snippet of DataFrame
data_files = {
    'train': '/kaggle/input/nlp-getting-started/train.csv',
    'test': '/kaggle/input/nlp-getting-started/test.csv'
}

nlp_dataset = load_dataset('csv', data_files=data_files)

In [ ]:
# Dataset structure 
nlp_dataset

In [ ]:
# Total tweets per target
nlp_train_df = nlp_dataset['train'].to_pandas()

sns.countplot(nlp_train_df, x='target')
plt.xticks(ticks=[0, 1], labels=['Non-Disaster', 'Disaster'])
plt.show()

In [ ]:
# Create bag of words for disaster tweets
disaster_tweets = nlp_train_df[nlp_train_df['target'] == 1]

cv = CountVectorizer(stop_words='english')
disaster_tweets_matrix = cv.fit_transform(disaster_tweets['text'])
disaster_freqs = zip(cv.get_feature_names_out(), disaster_tweets_matrix.sum(axis=0).tolist()[0]) 

print(sorted(disaster_freqs, key=lambda x: -x[1])[:20])

In [ ]:
# Word cloud for disaster tweets
disaster_tweets_text = " ".join(tweet for tweet in disaster_tweets['text'])
disaster_tweets_wc = WordCloud().generate(disaster_tweets_text)
plt.imshow(disaster_tweets_wc)
plt.axis('off')
plt.show()

In [ ]:
# Create bag of words for Non-Disaster Tweets
non_disaster_tweets = nlp_train_df[nlp_train_df['target'] == 0]

non_disaster_tweets_matrix = cv.fit_transform(non_disaster_tweets['text'])
non_disaster_freqs = zip(cv.get_feature_names_out(), disaster_tweets_matrix.sum(axis=0).tolist()[0]) 

print(sorted(non_disaster_freqs, key=lambda x: -x[1])[:20])

In [ ]:
# Word cloud for non-disaster tweets
non_disaster_tweets_text = " ".join(tweet for tweet in non_disaster_tweets['text'])
non_disaster_tweets_wc = WordCloud().generate(non_disaster_tweets_text)
plt.imshow(non_disaster_tweets_wc)
plt.axis('off')
plt.show()

In [ ]:
# Check sample data
nlp_dataset['train'][0]

In [ ]:
# Pre-processing (tokenization)
checkpoint = 'bert-large-uncased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint, do_lower_case=True)

def tokenize_function(dataset):
    return tokenizer(dataset['text'], truncation=True,
                    padding='max_length', max_length=128)


tokenized_datasets = nlp_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_datasets

In [ ]:
# Prepare for training
tokenized_datasets = tokenized_datasets.remove_columns(["id", "keyword", "location", "text"])
tokenized_datasets = tokenized_datasets.rename_column("target", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets

In [ ]:
tokenized_datasets_clean = tokenized_datasets["train"].train_test_split(train_size=0.8, seed=42)
tokenized_datasets_clean["validation"] = tokenized_datasets_clean.pop("test")

# Add the "test" set to our `DatasetDict`
tokenized_datasets_clean["test"] = tokenized_datasets["test"]
tokenized_datasets_clean

In [ ]:
# Create train and eval data loaders
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets_clean["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets_clean["validation"], batch_size=8, collate_fn=data_collator
)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

In [ ]:
# Model
def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    elif torch.backends.mps.is_available():
        return 'mps'
    else:
        return 'cpu'


id2label = {0: "Non-Disaster", 1: "Disaster"}
label2id = {"Non-Disaster": 0, "Disaster": 1}
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, 
                                                           num_labels=2,
                                                           id2label=id2label, 
                                                           label2id=label2id)

device = get_device()
model = model.to(device)

In [ ]:
# Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

print(f'Number of training steps: {num_training_steps}')

In [ ]:
# Training Loop
metric = evaluate.load("f1")

for epoch in range(num_epochs):
    
    train_loss = 0
    
    model.train()
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        train_loss += loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
    model.eval()
    for batch in eval_dataloader:
        with torch.inference_mode():
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch["labels"])
        
    print(f'\nEpoch: {epoch+1}')
    print(f'Train loss: {train_loss.item()/len(train_dataloader)}')
    print(f'Eval metrics: {metric.compute()}')

In [ ]:
submission = pd.DataFrame()
submission['id'] = nlp_dataset['test']['id']

tokenized_datasets_test = tokenized_datasets_clean["test"] 
tokenized_datasets_test = tokenized_datasets_test.remove_columns(["labels"])

test_dataloader = DataLoader(
    tokenized_datasets_test, batch_size=8, collate_fn=data_collator
)

predictions = []

model.eval()
for batch in test_dataloader:
    with torch.inference_mode():
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
    
        logits = outputs.logits
        predictions.append(torch.argmax(logits, dim=-1))

all_preds = torch.cat(predictions).cpu().numpy()
all_preds

submission['target'] = all_preds
submission.to_csv('submission.csv', index=False)